#### Deletion Vectors
Deletion vectors are a storage optimization feature that accelerates modifications to tables. By default, deleting a single row requires rewriting the entire Parquet file containing that record. Deletion vectors avoid this overhead. When deletion vectors are enabled, DELETE, UPDATE, and MERGE operations mark rows as modified without rewriting the Parquet file. Reads then resolve the current table state by applying the modifications recorded in deletion vectors.

#### Enable deletion vectors
```sql
-- For Delta tables
CREATE TABLE <table-name> [options] TBLPROPERTIES ('delta.enableDeletionVectors' = true);

ALTER TABLE <table-name> SET TBLPROPERTIES ('delta.enableDeletionVectors' = true);
```

In [ ]:
%%bash

## Delete Existing Delta Table /invoices_cow
aws --endpoint-url http://minio.sandbox.net:9010 s3 ls s3://warehouse/default/deltalake/invoices_cow --recursive

aws --endpoint-url http://minio.sandbox.net:9010 s3 ls s3://warehouse/default/deltalake/invoices_mor --recursive

2026-04-01 12:45:02       2284 default/deltalake/invoices_cow/_delta_log/00000000000000000000.crc
2026-04-01 12:45:01       2607 default/deltalake/invoices_cow/_delta_log/00000000000000000000.json
2026-04-01 12:45:16       2284 default/deltalake/invoices_cow/_delta_log/00000000000000000001.crc
2026-04-01 12:45:15       1735 default/deltalake/invoices_cow/_delta_log/00000000000000000001.json
2026-04-01 12:45:30       2282 default/deltalake/invoices_cow/_delta_log/00000000000000000002.crc
2026-04-01 12:45:29       1733 default/deltalake/invoices_cow/_delta_log/00000000000000000002.json
2026-04-01 12:44:58          0 default/deltalake/invoices_cow/_delta_log/_commits/
2026-04-01 12:45:00       5745 default/deltalake/invoices_cow/part-00000-4ae32836-f407-4d1c-a8d3-bf9664e70c1e-c000.snappy.parquet
2026-04-01 12:45:15       5745 default/deltalake/invoices_cow/part-00000-640a41dd-27d5-49a7-bd7e-23e39a56490f-c000.snappy.parquet
2026-04-01 12:45:29       5722 default/deltalake/invoices_cow/part

In [ ]:
%%bash

## Delete Existing Delta Table /invoices_cow
aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/invoices_cow --recursive

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/invoices_mor --recursive

In [4]:
%load_ext sql

In [5]:
%sql spark

### CoW (Copy-on-Write)

In [6]:
%%sql

SELECT * FROM 
PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
LIMIT 5;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11
101,I302068,Female,58,Shoes,2,1200.34,Debit Card,2022-07-15,Metropol AVM,None
102,I193229,Male,40,Clothing,5,1500.4,Cash,2021-03-05,Cevahir AVM,None
103,I313092,Female,23,Cosmetics,2,81.32,Credit Card,2022-04-23,Zorlu Center,None
104,I258750,Female,40,Food & Beverage,3,15.69,Cash,2022-04-04,Istinye Park,None
105,I126182,Female,57,Clothing,5,1500.4,Cash,2022-02-06,Istinye Park,None


In [8]:
%%sql

CREATE OR REPLACE TABLE spark_catalog.deltalake.invoices_cow 
USING DELTA
TBLPROPERTIES ('delta.enableDeletionVectors' = false)
AS 
SELECT * FROM 
PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`;

Running query in 'SparkSession'

++
||
++
++

In [9]:
%%sql

DESCRIBE EXTENDED spark_catalog.deltalake.invoices_cow;

Running query in 'SparkSession'

19 rows affected.

Field 1,Field 2,Field 3
customer_id,int,None
invoice_no,string,None


In [10]:
%%sql

UPDATE spark_catalog.deltalake.invoices_cow 
SET age = 55 
WHERE customer_id = 105;

Running query in 'SparkSession'

1 rows affected.

Field 1
1


In [11]:
%%sql

DESCRIBE HISTORY spark_catalog.deltalake.invoices_cow;

Running query in 'SparkSession'

2 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
1,2026-04-01 12:45:15,None,None,UPDATE,"{'predicate': '[""(customer_id#847 = 105)""]'}",None,None,None,0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '1772', 'numDeletionVectorsRemoved': '0', 'numUpdatedRows': '1', 'numRemovedFiles': '1', 'rewriteTimeMs': '207', 'numRemovedBytes': '5745', 'scanTimeMs': '1563', 'numCopiedRows': '99', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numAddedBytes': '5745'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-01 12:45:01,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{""delta.enableDeletionVectors"":""false""}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,False,"{'numOutputRows': '100', 'numOutputBytes': '5745', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [12]:
%%sql

DELETE FROM spark_catalog.deltalake.invoices_cow 
WHERE customer_id = 102;

Running query in 'SparkSession'

1 rows affected.

Field 1
1


In [13]:
%%sql

DESCRIBE HISTORY spark_catalog.deltalake.invoices_cow;

Running query in 'SparkSession'

3 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
2,2026-04-01 12:45:29,None,None,DELETE,"{'predicate': '[""(customer_id#2668 = 102)""]'}",None,None,None,1,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '1119', 'numDeletionVectorsRemoved': '0', 'numRemovedFiles': '1', 'rewriteTimeMs': '191', 'numRemovedBytes': '5745', 'scanTimeMs': '927', 'numCopiedRows': '99', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numDeletedRows': '1', 'numAddedBytes': '5722'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
1,2026-04-01 12:45:15,None,None,UPDATE,"{'predicate': '[""(customer_id#847 = 105)""]'}",None,None,None,0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '1772', 'numDeletionVectorsRemoved': '0', 'numUpdatedRows': '1', 'numRemovedFiles': '1', 'rewriteTimeMs': '207', 'numRemovedBytes': '5745', 'scanTimeMs': '1563', 'numCopiedRows': '99', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numAddedBytes': '5745'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-01 12:45:01,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{""delta.enableDeletionVectors"":""false""}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,False,"{'numOutputRows': '100', 'numOutputBytes': '5745', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


### MoR (Merge-on-Read)

In [14]:
%%sql

CREATE OR REPLACE TABLE spark_catalog.deltalake.invoices_mor
USING DELTA
TBLPROPERTIES ('delta.enableDeletionVectors' = true)
AS 
SELECT * FROM 
PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`;

Running query in 'SparkSession'

++
||
++
++

In [15]:
%%sql

DESCRIBE EXTENDED spark_catalog.deltalake.invoices_mor;

Running query in 'SparkSession'

19 rows affected.

Field 1,Field 2,Field 3
customer_id,int,None
invoice_no,string,None


In [16]:
%%sql

DELETE FROM spark_catalog.deltalake.invoices_mor
WHERE customer_id = 102;

Running query in 'SparkSession'

1 rows affected.

Field 1
1


In [17]:
%%sql

DESCRIBE HISTORY spark_catalog.deltalake.invoices_mor;

Running query in 'SparkSession'

2 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
1,2026-04-01 12:46:10,None,None,DELETE,"{'predicate': '[""(customer_id#5170 = 102)""]'}",None,None,None,0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '0', 'executionTimeMs': '1481', 'numDeletionVectorsRemoved': '0', 'numRemovedFiles': '0', 'rewriteTimeMs': '0', 'numRemovedBytes': '0', 'scanTimeMs': '0', 'numCopiedRows': '0', 'numDeletionVectorsAdded': '1', 'numAddedChangeFiles': '0', 'numDeletedRows': '1', 'numAddedBytes': '0'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-01 12:45:59,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{""delta.enableDeletionVectors"":""true""}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,False,"{'numOutputRows': '100', 'numOutputBytes': '5745', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [18]:
%%sql

UPDATE spark_catalog.deltalake.invoices_mor
SET age = 55 
WHERE customer_id = 105;

Running query in 'SparkSession'

1 rows affected.

Field 1
1


In [19]:
%%sql

DESCRIBE HISTORY spark_catalog.deltalake.invoices_mor;

Running query in 'SparkSession'

3 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
2,2026-04-01 12:46:22,None,None,UPDATE,"{'predicate': '[""(customer_id#7478 = 105)""]'}",None,None,None,1,Serializable,False,"{'numDeletionVectorsUpdated': '1', 'numAddedFiles': '1', 'executionTimeMs': '1322', 'numDeletionVectorsRemoved': '1', 'numUpdatedRows': '1', 'numRemovedFiles': '1', 'rewriteTimeMs': '171', 'numRemovedBytes': '0', 'scanTimeMs': '937', 'numCopiedRows': '0', 'numDeletionVectorsAdded': '1', 'numAddedChangeFiles': '0', 'numAddedBytes': '3076'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
1,2026-04-01 12:46:10,None,None,DELETE,"{'predicate': '[""(customer_id#5170 = 102)""]'}",None,None,None,0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '0', 'executionTimeMs': '1481', 'numDeletionVectorsRemoved': '0', 'numRemovedFiles': '0', 'rewriteTimeMs': '0', 'numRemovedBytes': '0', 'scanTimeMs': '0', 'numCopiedRows': '0', 'numDeletionVectorsAdded': '1', 'numAddedChangeFiles': '0', 'numDeletedRows': '1', 'numAddedBytes': '0'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-01 12:45:59,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{""delta.enableDeletionVectors"":""true""}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,False,"{'numOutputRows': '100', 'numOutputBytes': '5745', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [20]:
%%sql

OPTIMIZE spark_catalog.deltalake.invoices_mor;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/invoices_mor,"Row(numFilesAdded=1, numFilesRemoved=2, filesAdded=Row(min=5734, max=5734, avg=5734.0, totalFiles=1, totalSize=5734), filesRemoved=Row(min=3076, max=5745, avg=4410.5, totalFiles=2, totalSize=8821), partitionsOptimized=1, zOrderStats=None, clusteringStats=None, numBins=1, numBatches=1, totalConsideredFiles=2, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1775027792839, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=Row(numDeletionVectorsRemoved=1, numDeletionVectorRowsRemoved=2), numTableColumns=11, numTableColumnsWithStats=11)"


In [21]:
%%sql

SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM spark_catalog.deltalake.invoices_mor RETAIN 0 HOURS;

Running query in 'SparkSession'

1 rows affected.

Deleted 12 files and directories in a total of 1 directories.


1 rows affected.

Field 1
s3a://warehouse/default/deltalake/invoices_mor
